In [1]:
"""
=============================================================================
  DigitalOcean Docs — Full Algorithmic Pipeline
  ─────────────────────────────────────────────
  1. Fine-tuned embedding model  (nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8
     loaded via Hugging Face, NOT downloaded locally)
  2. AI Agent — Semantic Search Engine + Customer Support / RAG Chatbot
     • Semantic search with vector index + re-ranking
     • Customer support RAG with multi-agent routing
     • DigitalOcean Gradient full-stack feature mirroring

  Pipeline stages (per spec):
    1. Data Preparation
    2. Data Splitting & Evaluation Setup (40/15/15/30)
    3. Feature Engineering
    4. Model Development  (best accuracy model saved)
    5. Model Saving & Early Stopping
    6. Ensembling & Hybridization
    7. Resource Management
    8. Model Evaluation & Visualization
    9. Sophisticated Feature Engineering
   10. EDA + full graphical plots (plt.show + plt.savefig)
   11. Anti-Overfitting Techniques
=============================================================================
"""

# ─── 0. IMPORTS & GLOBAL CONFIG ──────────────────────────────────────────────

import os, gc, re, sys, json, time, logging, warnings, hashlib, pickle
from pathlib import Path
from typing import List, Dict, Optional, Tuple, Any
import numpy as np
import pandas as pd
import psutil, platform
from tqdm import tqdm
import matplotlib
matplotlib.use("Agg")          # non-interactive backend (memory safe)
import matplotlib.pyplot as plt
import seaborn as sns

# --- scikit-learn ---
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RandomizedSearchCV
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)
from sklearn.ensemble import (
    GradientBoostingClassifier, RandomForestClassifier,
    VotingClassifier, BaggingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
import scipy.sparse as sp

# --- PyTorch ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from torch.optim.lr_scheduler import ReduceLROnPlateau

# --- HuggingFace Transformers / PEFT ---
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForCausalLM,
    TrainingArguments, Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
try:
    from peft import (
        get_peft_model, LoraConfig, TaskType,
        prepare_model_for_kbit_training,
    )
    PEFT_AVAILABLE = True
except ImportError:
    PEFT_AVAILABLE = False

# --- HTML / web parsing ---
from bs4 import BeautifulSoup

# --- FAISS vector index ---
try:
    import faiss as _faiss_lib
    faiss = _faiss_lib          # module-level alias, always accessible
    FAISS_AVAILABLE = True
except ImportError:
    faiss = None                # explicit None so references do not NameError
    FAISS_AVAILABLE = False

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# ─── GLOBAL PATHS ────────────────────────────────────────────────────────────

ROOT_PATH       = Path("/kaggle/input/datasets/tobimichigan/digitalocreandoc-db")
HF_TOKEN_PATH   = Path("/kaggle/input/datasets/tobimichigan/hf-tokens/hf_tokens/hf.token.txt")
OUTPUT_DIR      = Path("./outputs");          OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR       = Path("./fine_tuned_model"); MODEL_DIR.mkdir(exist_ok=True)
PLOTS_DIR       = Path("./plots");            PLOTS_DIR.mkdir(exist_ok=True)
INDEX_DIR       = Path("./vector_index");     INDEX_DIR.mkdir(exist_ok=True)

# ─── MEMORY CONFIG ───────────────────────────────────────────────────────────

MEMORY_LIMIT_GB   = 6.0      # stop ops before this threshold
CHUNK_ROWS        = 10_000   # max rows per chunk
MAX_CHUNKS        = 3
SAMPLE_RATE       = 0.00005  # ultra-conservative sampling
EMBED_DIM         = 384      # fallback embedding dim


# =============================================================================
# SECTION 1 — UTILITY & RESOURCE MANAGEMENT
# =============================================================================

def get_memory_gb() -> float:
    return psutil.Process(os.getpid()).memory_info().rss / 1e9

def log_memory(tag: str = ""):
    used = get_memory_gb()
    avail = psutil.virtual_memory().available / 1e9
    logger.info(f"[MEM {tag}] used={used:.2f}GB  avail={avail:.2f}GB")

def memory_safe() -> bool:
    return get_memory_gb() < MEMORY_LIMIT_GB

def force_cleanup(*vars_to_del):
    """Aggressive garbage collection."""
    for v in vars_to_del:
        try:
            del v
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    log_memory("after_cleanup")

def load_hf_token() -> Optional[str]:
    """Load HuggingFace token from file (never hardcoded)."""
    try:
        token = HF_TOKEN_PATH.read_text().strip()
        logger.info("HF token loaded successfully.")
        return token
    except Exception as e:
        logger.warning(f"Could not load HF token: {e}")
        return None

def configure_torch():
    """Memory-safe torch + GPU config."""
    if torch.cuda.is_available():
        torch.cuda.set_per_process_memory_fraction(0.7)
        logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        logger.info("Running on CPU.")
    torch.backends.cudnn.benchmark = True

configure_torch()


# =============================================================================
# SECTION 2 — DATA PREPARATION: HTML CRAWLER & PARSER
# =============================================================================

class HTMLDocumentCrawler:
    """
    Recursively crawl ROOT_PATH, parse all .html files,
    and return a DataFrame with [path, url_slug, title, body_text,
    doc_type, word_count].
    Memory-safe: streams files in chunks.
    """

    CHUNK_SIZE = 500   # files per chunk

    def __init__(self, root: Path = ROOT_PATH):
        self.root = root
        self.records: List[Dict] = []

    def _classify_doc(self, path_str: str) -> str:
        p = path_str.lower()
        if "tutorial" in p:   return "tutorial"
        if "support" in p:    return "support"
        if "community" in p:  return "community"
        if "blog" in p:       return "blog"
        if "docs" in p:       return "docs"
        if "product" in p:    return "product"
        return "general"

    def _parse_html(self, filepath: Path) -> Optional[Dict]:
        try:
            raw = filepath.read_text(encoding="utf-8", errors="ignore")
            soup = BeautifulSoup(raw, "html.parser")
            title = soup.title.string.strip() if soup.title else filepath.stem
            # remove nav, header, footer noise
            for tag in soup(["script","style","nav","footer","header","aside"]):
                tag.decompose()
            body = " ".join(soup.get_text(separator=" ").split())
            return {
                "path":       str(filepath.relative_to(self.root)),
                "url_slug":   filepath.stem,
                "title":      title[:512],
                "body_text":  body[:8000],   # cap at 8k chars
                "doc_type":   self._classify_doc(str(filepath)),
                "word_count": len(body.split()),
            }
        except Exception as e:
            logger.debug(f"Parse error {filepath}: {e}")
            return None

    def crawl(self) -> pd.DataFrame:
        html_files = sorted(self.root.rglob("*.html"))
        logger.info(f"Found {len(html_files)} HTML files under {self.root}")

        all_records: List[Dict] = []
        chunks = [html_files[i:i+self.CHUNK_SIZE]
                  for i in range(0, len(html_files), self.CHUNK_SIZE)]

        for chunk_idx, chunk in enumerate(
                tqdm(chunks, desc="Crawling HTML chunks")):
            if not memory_safe():
                logger.warning("Memory limit reached — stopping crawl early.")
                break
            recs = []
            for fp in tqdm(chunk, desc=f"  Chunk {chunk_idx}", leave=False):
                rec = self._parse_html(fp)
                if rec:
                    recs.append(rec)
            all_records.extend(recs)
            log_memory(f"chunk_{chunk_idx}")
            gc.collect()

        df = pd.DataFrame(all_records)
        logger.info(f"Crawled {len(df)} valid documents.")
        return df

    @staticmethod
    def preprocess(df: pd.DataFrame) -> pd.DataFrame:
        """Cleaning, missing-value handling, type normalisation."""
        logger.info("Preprocessing documents...")
        df = df.dropna(subset=["body_text"]).copy()
        df["body_text"]  = df["body_text"].str.strip()
        df["title"]      = df["title"].fillna(df["url_slug"])
        df["word_count"] = pd.to_numeric(df["word_count"], errors="coerce").fillna(0).astype(int)
        # label-encode doc_type
        le = LabelEncoder()
        df["doc_type_id"] = le.fit_transform(df["doc_type"])
        df["label"]       = df["doc_type_id"]   # classification target
        logger.info(f"After preprocessing: {df.shape}")
        return df, le


# =============================================================================
# SECTION 3 — DATA SPLITTING (40/15/15/30)
# =============================================================================

class DataSplitter:
    """
    Four-way stratified split:
      Train=40%  Val=15%  Test=15%  Holdout=30%
    Strict separation: holdout never used in training or validation.
    """

    def __init__(self, df: pd.DataFrame, label_col: str = "label",
                 random_state: int = 42):
        self.df    = df.reset_index(drop=True)
        self.label = label_col
        self.rs    = random_state

    def split(self):
        y = self.df[self.label]
        # Step 1: carve out holdout (30%)
        idx_main, idx_holdout = train_test_split(
            self.df.index, test_size=0.30, random_state=self.rs,
            stratify=y)
        # Step 2: of remaining 70% → train:val:test = 40:15:15  → ~57:21:21
        remaining = self.df.loc[idx_main]
        y_rem = remaining[self.label]
        idx_train, idx_valtest = train_test_split(
            remaining.index, test_size=0.4286, random_state=self.rs,
            stratify=y_rem)
        idx_val, idx_test = train_test_split(
            idx_valtest, test_size=0.50, random_state=self.rs,
            stratify=self.df.loc[idx_valtest, self.label])

        splits = {
            "train":   self.df.loc[idx_train].reset_index(drop=True),
            "val":     self.df.loc[idx_val].reset_index(drop=True),
            "test":    self.df.loc[idx_test].reset_index(drop=True),
            "holdout": self.df.loc[idx_holdout].reset_index(drop=True),
        }
        for name, s in splits.items():
            logger.info(f"  {name:8s}: {len(s):5d} rows  "
                        f"({100*len(s)/len(self.df):.1f}%)")
        return splits

    @staticmethod
    def verify_data_splits(splits: Dict) -> bool:
        """Ensure no index overlap between splits."""
        all_indices = []
        for name, s in splits.items():
            all_indices.append(set(s.index.tolist()))
        for i, (n1, s1) in enumerate(splits.items()):
            for j, (n2, s2) in enumerate(splits.items()):
                if i >= j:
                    continue
                overlap = set(s1.index) & set(s2.index)
                if overlap:
                    logger.error(f"DATA LEAK between {n1} and {n2}: {len(overlap)} rows")
                    return False
        logger.info("Split verification PASSED — no data leakage.")
        return True

    @staticmethod
    def check_domain_relevance(splits: Dict, label_col: str = "label"):
        """Confirm label distribution is consistent across splits."""
        for name, s in splits.items():
            dist = s[label_col].value_counts(normalize=True).to_dict()
            logger.info(f"  {name} label dist: "
                        + ", ".join(f"{k}:{v:.2f}" for k, v in dist.items()))


# =============================================================================
# SECTION 4 — FEATURE ENGINEERING
# =============================================================================

class FeatureEngineer:
    """
    TF-IDF + SVD (LSA) + hand-crafted meta features.
    Returns scipy sparse or dense numpy arrays.
    """

    def __init__(self, max_features: int = 20_000, n_components: int = 128):
        self.tfidf = TfidfVectorizer(
            max_features=max_features,
            sublinear_tf=True,
            ngram_range=(1, 2),
            strip_accents="unicode",
            analyzer="word",
            min_df=2,
        )
        self.svd   = TruncatedSVD(n_components=n_components, random_state=42)
        self.scaler = StandardScaler()

    def _meta_features(self, df: pd.DataFrame) -> np.ndarray:
        feats = np.column_stack([
            df["word_count"].values,
            df["body_text"].str.len().values,
            df["body_text"].str.count(r"\bcode\b").values,
            df["body_text"].str.count(r"\bapi\b").values,
            df["body_text"].str.count(r"\btutorial\b").values,
            df["url_slug"].str.count(r"-").values,
        ]).astype(np.float32)
        return feats

    def fit_transform(self, df_train: pd.DataFrame):
        logger.info("Feature engineering: fit on training set...")
        tfidf_mat = self.tfidf.fit_transform(df_train["body_text"])
        lsa_mat   = self.svd.fit_transform(tfidf_mat)
        meta      = self._meta_features(df_train)
        meta_scaled = self.scaler.fit_transform(meta)
        X = np.hstack([lsa_mat, meta_scaled])
        logger.info(f"Train feature matrix: {X.shape}")
        force_cleanup(tfidf_mat)
        return X

    def transform(self, df: pd.DataFrame):
        tfidf_mat = self.tfidf.transform(df["body_text"])
        lsa_mat   = self.svd.transform(tfidf_mat)
        meta      = self._meta_features(df)
        meta_scaled = self.scaler.transform(meta)
        X = np.hstack([lsa_mat, meta_scaled])
        force_cleanup(tfidf_mat)
        return X

    def pca_plot(self, X: np.ndarray, y: np.ndarray, tag: str = "train"):
        """2-D PCA scatter for EDA."""
        pca  = PCA(n_components=2, random_state=42)
        proj = pca.fit_transform(X[:3000])
        fig, ax = plt.subplots(figsize=(8, 6))
        sc = ax.scatter(proj[:,0], proj[:,1], c=y[:3000],
                        cmap="tab10", alpha=0.5, s=10)
        plt.colorbar(sc, ax=ax, label="Doc type")
        ax.set_title(f"PCA 2D — {tag} set")
        path = PLOTS_DIR / f"pca_{tag}.png"
        plt.savefig(path, dpi=120, bbox_inches="tight")
        plt.show(); plt.close()
        logger.info(f"Saved {path}")


# =============================================================================
# SECTION 5 — EDA & VISUALISATION
# =============================================================================

def run_eda(df: pd.DataFrame, splits: Dict):
    logger.info("Running EDA...")

    # 1. Document type distribution
    fig, ax = plt.subplots(figsize=(9, 4))
    df["doc_type"].value_counts().plot(kind="bar", ax=ax, color="steelblue")
    ax.set_title("Document Type Distribution (full corpus)")
    ax.set_xlabel("Doc Type"); ax.set_ylabel("Count")
    plt.tight_layout()
    p = PLOTS_DIR / "eda_doc_type_dist.png"
    plt.savefig(p, dpi=120); plt.show(); plt.close()

    # 2. Word count distribution
    fig, ax = plt.subplots(figsize=(9, 4))
    df["word_count"].clip(upper=5000).hist(bins=60, ax=ax, color="teal")
    ax.set_title("Word Count Distribution")
    ax.set_xlabel("Words"); ax.set_ylabel("Frequency")
    plt.tight_layout()
    p = PLOTS_DIR / "eda_word_count.png"
    plt.savefig(p, dpi=120); plt.show(); plt.close()

    # 3. Split size comparison
    split_sizes = {k: len(v) for k, v in splits.items()}
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(split_sizes.keys(), split_sizes.values(), color=["#4C72B0","#DD8452","#55A868","#C44E52"])
    ax.set_title("Data Split Sizes")
    ax.set_ylabel("N rows")
    plt.tight_layout()
    p = PLOTS_DIR / "eda_split_sizes.png"
    plt.savefig(p, dpi=120); plt.show(); plt.close()

    # 4. Correlation heatmap of numeric cols
    numeric = df[["word_count","doc_type_id"]].corr()
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(numeric, annot=True, ax=ax, cmap="coolwarm")
    ax.set_title("Numeric Feature Correlation")
    plt.tight_layout()
    p = PLOTS_DIR / "eda_correlation.png"
    plt.savefig(p, dpi=120); plt.show(); plt.close()

    force_cleanup()
    logger.info("EDA complete.")


# =============================================================================
# SECTION 6 — CLASSICAL ML ENSEMBLE (Gradient Boosting + RF + LR)
# =============================================================================

class ClassicalEnsemble:
    """
    Soft-voting ensemble: GBM + RandomForest + LogReg.
    Hyperparameter tuning via RandomizedSearchCV.
    """

    def __init__(self):
        self.best_model = None
        self.best_acc   = 0.0
        self.history    = {}

    def _build_estimators(self):
        gbm = GradientBoostingClassifier(
            n_estimators=200, learning_rate=0.05,
            max_depth=5, subsample=0.8,
            min_samples_leaf=5, random_state=42,
            validation_fraction=0.1, n_iter_no_change=15,
            tol=1e-4,
        )
        rf  = RandomForestClassifier(
            n_estimators=200, max_depth=12,
            min_samples_leaf=4, n_jobs=-1,
            random_state=42, class_weight="balanced",
        )
        lr  = LogisticRegression(
            max_iter=500, C=1.0, class_weight="balanced",
            solver="lbfgs", n_jobs=-1,
        )
        return gbm, rf, lr

    def fit(self, X_train, y_train, X_val, y_val):
        logger.info("Training classical ensemble...")
        gbm, rf, lr = self._build_estimators()

        # Hyperparameter search on RF (as representative)
        param_dist = {
            "n_estimators": [100, 200, 300],
            "max_depth":    [8, 12, 16, None],
            "min_samples_leaf": [2, 4, 8],
        }
        rs = RandomizedSearchCV(
            rf, param_dist, n_iter=8, cv=3, scoring="accuracy",
            random_state=42, n_jobs=-1, verbose=0
        )
        with tqdm(total=1, desc="RandomizedSearchCV RF"):
            rs.fit(X_train, y_train)
        best_rf = rs.best_estimator_
        logger.info(f"Best RF params: {rs.best_params_}")

        vote = VotingClassifier(
            estimators=[("gbm", gbm), ("rf", best_rf), ("lr", lr)],
            voting="soft", n_jobs=-1,
        )
        with tqdm(total=1, desc="Fitting VotingClassifier"):
            vote.fit(X_train, y_train)

        val_acc = accuracy_score(y_val, vote.predict(X_val))
        logger.info(f"Ensemble val accuracy: {val_acc:.4f}")
        self.best_model = vote
        self.best_acc   = val_acc
        self.history["val_acc"] = val_acc
        return vote

    def evaluate(self, X, y, tag: str = "test"):
        preds = self.best_model.predict(X)
        proba = (self.best_model.predict_proba(X)
                 if hasattr(self.best_model, "predict_proba") else None)
        acc = accuracy_score(y, preds)
        report = classification_report(y, preds, zero_division=0)
        logger.info(f"\n[{tag}] Accuracy: {acc:.4f}\n{report}")
        return acc, preds, proba

    def save(self, path: Path = OUTPUT_DIR / "classical_ensemble.pkl"):
        with open(path, "wb") as f:
            pickle.dump(self.best_model, f)
        logger.info(f"Classical ensemble saved → {path}")

    @staticmethod
    def load(path: Path = OUTPUT_DIR / "classical_ensemble.pkl"):
        with open(path, "rb") as f:
            return pickle.load(f)


# =============================================================================
# SECTION 7 — DEEP MLP CLASSIFIER (PyTorch)
# =============================================================================

class DeepMLP(nn.Module):
    """
    Micro-sized MLP with:
      • BatchNorm after each layer
      • Dropout up to 40%
      • Gaussian noise on input (augmentation)
      • L2 weight decay (set in optimizer)
    """

    def __init__(self, input_dim: int, num_classes: int,
                 hidden_dims=(256, 128, 64), dropout=0.40):
        super().__init__()
        layers = []
        in_d = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(in_d, h),
                nn.BatchNorm1d(h),
                nn.GELU(),
                nn.Dropout(dropout),
            ]
            in_d = h
        layers.append(nn.Linear(in_d, num_classes))
        self.net   = nn.Sequential(*layers)
        self.noise_std = 0.02

    def forward(self, x):
        if self.training:
            x = x + torch.randn_like(x) * self.noise_std
        return self.net(x)


def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0))
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


class MLPTrainer:
    """Trainer with early stopping, ReduceLROnPlateau, mixup, class weights."""

    def __init__(self, model: nn.Module, num_classes: int,
                 device: str = None, label_smoothing: float = 0.1):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model  = model.to(self.device)
        self.num_classes = num_classes
        self.best_acc  = 0.0
        self.best_state = None
        self.train_acc_hist: List[float] = []
        self.val_acc_hist:   List[float] = []
        self.label_smoothing = label_smoothing

    def _make_loader(self, X, y, batch_size=256, shuffle=True,
                     class_weights=None):
        Xt = torch.tensor(X, dtype=torch.float32)
        yt = torch.tensor(y, dtype=torch.long)
        ds = TensorDataset(Xt, yt)
        sampler = None
        if shuffle and class_weights is not None:
            w = class_weights[y]
            sampler = WeightedRandomSampler(
                torch.tensor(w, dtype=torch.float32), len(w))
            shuffle = False
        return DataLoader(ds, batch_size=batch_size,
                          shuffle=shuffle, sampler=sampler,
                          pin_memory=True, num_workers=0)

    def _compute_class_weights(self, y):
        classes, counts = np.unique(y, return_counts=True)
        w = len(y) / (len(classes) * counts)
        cw = np.zeros(self.num_classes)
        for c, wi in zip(classes, w):
            cw[c] = wi
        return cw

    def train(self, X_train, y_train, X_val, y_val,
              epochs=40, lr=1e-3, weight_decay=1e-4,
              patience=8, min_delta=1e-4, use_mixup=True):

        cw = self._compute_class_weights(y_train)
        cw_tensor = torch.tensor(cw, dtype=torch.float32).to(self.device)
        train_loader = self._make_loader(X_train, y_train, class_weights=cw)
        val_loader   = self._make_loader(X_val, y_val, shuffle=False)

        criterion = nn.CrossEntropyLoss(
            weight=cw_tensor, label_smoothing=self.label_smoothing)
        optimizer = torch.optim.AdamW(
            self.model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = ReduceLROnPlateau(
            optimizer, mode="max", patience=4, factor=0.5)

        no_improve = 0
        for epoch in tqdm(range(epochs), desc="MLP Training"):
            # — train —
            self.model.train()
            for Xb, yb in train_loader:
                Xb, yb = Xb.to(self.device), yb.to(self.device)
                if use_mixup:
                    Xb, ya, yb2, lam = mixup_data(Xb, yb)
                    ya, yb2 = ya.to(self.device), yb2.to(self.device)
                optimizer.zero_grad()
                logits = self.model(Xb)
                if use_mixup:
                    loss = lam * criterion(logits, ya) + (1-lam) * criterion(logits, yb2)
                else:
                    loss = criterion(logits, yb)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()

            # — validate —
            val_acc = self._eval(val_loader)
            self.model.train()
            train_acc = self._eval(self._make_loader(
                X_train, y_train, shuffle=False))
            self.model.train()

            self.train_acc_hist.append(train_acc)
            self.val_acc_hist.append(val_acc)
            scheduler.step(val_acc)

            if val_acc > self.best_acc + min_delta:
                self.best_acc   = val_acc
                self.best_state = {k: v.cpu().clone()
                                   for k, v in self.model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1

            if no_improve >= patience:
                logger.info(f"Early stop at epoch {epoch+1}  "
                            f"best_val_acc={self.best_acc:.4f}")
                break

        # restore best weights
        if self.best_state:
            self.model.load_state_dict(self.best_state)
        logger.info(f"MLP training done. Best val acc: {self.best_acc:.4f}")

    def _eval(self, loader) -> float:
        self.model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for Xb, yb in loader:
                Xb, yb = Xb.to(self.device), yb.to(self.device)
                preds = self.model(Xb).argmax(dim=1)
                correct += (preds == yb).sum().item()
                total   += len(yb)
        self.model.train()
        return correct / total if total else 0.0

    def predict(self, X) -> np.ndarray:
        self.model.eval()
        loader = self._make_loader(X, np.zeros(len(X), dtype=int),
                                   shuffle=False)
        all_preds = []
        with torch.no_grad():
            for Xb, _ in loader:
                preds = self.model(Xb.to(self.device)).argmax(dim=1)
                all_preds.append(preds.cpu().numpy())
        return np.concatenate(all_preds)

    def predict_proba(self, X) -> np.ndarray:
        self.model.eval()
        loader = self._make_loader(X, np.zeros(len(X), dtype=int),
                                   shuffle=False)
        probs = []
        with torch.no_grad():
            for Xb, _ in loader:
                p = F.softmax(self.model(Xb.to(self.device)), dim=1)
                probs.append(p.cpu().numpy())
        return np.concatenate(probs)

    def save(self, path: Path = MODEL_DIR / "mlp_best.pt"):
        torch.save({
            "state_dict": self.best_state or self.model.state_dict(),
            "best_acc":   self.best_acc,
        }, path)
        logger.info(f"MLP saved → {path}")

    def plot_training_curves(self):
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(self.train_acc_hist, label="Train Acc", marker="o", ms=3)
        ax.plot(self.val_acc_hist,   label="Val Acc",   marker="s", ms=3)
        peak_val_idx = int(np.argmax(self.val_acc_hist))
        ax.axvline(peak_val_idx, color="red", linestyle="--",
                   label=f"Peak val={self.best_acc:.4f}")
        ax.set_title("MLP Training Curves"); ax.set_xlabel("Epoch")
        ax.set_ylabel("Accuracy"); ax.legend()
        plt.tight_layout()
        p = PLOTS_DIR / "mlp_training_curves.png"
        plt.savefig(p, dpi=120); plt.show(); plt.close()
        logger.info(f"Saved {p}")


# =============================================================================
# SECTION 8 — NEMOTRON MODEL LOADER (fine-tuned model + AI Agent backbone)
# =============================================================================

class EmbeddingModel:
    """
    Loads nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8 correctly as a
    CausalLM (its true architecture) via HF Hub with trust_remote_code=True
    and 4-bit quantisation so the full model fits in GPU memory without a
    full local download.

    Responsibilities:
      1. Fine-tuned model backbone  — LoRA adapters on top, saved to MODEL_DIR
      2. AI Agent backbone          — generate() used for RAG / chat responses
      3. Embedding extraction       — mean-pool over last hidden states for
                                      vector index construction

    Falls back to sentence-transformers/all-MiniLM-L6-v2 for embeddings only
    when the primary model cannot be loaded (e.g. no GPU / quota exceeded),
    but agent generation always tries the primary first.
    """

    PRIMARY_MODEL  = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8"
    FALLBACK_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

    # Nemotron chat template tokens
    SYSTEM_TAG = "<extra_id_0>System\n"
    USER_TAG   = "\n<extra_id_1>User\n"
    ASST_TAG   = "\n<extra_id_1>Assistant\n"

    def __init__(self, hf_token: Optional[str] = None):
        self.token      = hf_token
        self.tokenizer  = None
        self.model      = None
        self.model_name : str  = ""
        self.dim        : int  = EMBED_DIM
        self.is_causal  : bool = False   # True when primary CausalLM loaded

    # ── 1. Model loading ───────────────────────────────────────────────────

    def load(self):
        """
        Load strategy:
          a) Try Nemotron as AutoModelForCausalLM + trust_remote_code + 4-bit
             quantisation (BitsAndBytes NF4) — fits in ~24 GB GPU RAM.
          b) If BitsAndBytes unavailable, try bfloat16 with device_map=auto.
          c) On any failure fall back to all-MiniLM-L6-v2 for embeddings only.
        """
        logger.info(f"Loading tokenizer: {self.PRIMARY_MODEL}")
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(
                self.PRIMARY_MODEL,
                token=self.token,
                trust_remote_code=True,
                padding_side="left",   # required for batch generation
            )
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            self.model = self._load_causal_lm(self.PRIMARY_MODEL)
            self.model.eval()
            self.dim        = self.model.config.hidden_size
            self.model_name = self.PRIMARY_MODEL
            self.is_causal  = True
            logger.info(
                f"Nemotron loaded successfully. "
                f"hidden_size={self.dim}  "
                f"dtype={next(self.model.parameters()).dtype}"
            )
        except Exception as e:
            logger.warning(
                f"Primary model load failed: {e}\n"
                f"→ Falling back to {self.FALLBACK_MODEL} for embeddings."
            )
            self._load_fallback()

        # Apply LoRA fine-tuning adapters on top of primary model
        if PEFT_AVAILABLE and self.is_causal:
            self._apply_lora()

    def _load_causal_lm(self, model_id: str):
        """
        Try 4-bit NF4 first (most memory-efficient), then bf16, then fp32.
        All paths use trust_remote_code=True so NemotronH config is accepted.
        """
        # ── path A: 4-bit NF4 quantisation (preferred) ──
        try:
            from transformers import BitsAndBytesConfig
            bnb_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
            )
            logger.info("Attempting 4-bit NF4 load...")
            m = AutoModelForCausalLM.from_pretrained(
                model_id,
                token=self.token,
                trust_remote_code=True,
                quantization_config=bnb_cfg,
                device_map="auto",
                output_hidden_states=True,
            )
            logger.info("4-bit NF4 load succeeded.")
            return m
        except Exception as e_bnb:
            logger.warning(f"4-bit load failed ({e_bnb}). Trying bf16...")

        # ── path B: bfloat16, device_map=auto ──
        try:
            m = AutoModelForCausalLM.from_pretrained(
                model_id,
                token=self.token,
                trust_remote_code=True,
                torch_dtype=torch.bfloat16,
                device_map="auto",
                output_hidden_states=True,
            )
            logger.info("bf16 load succeeded.")
            return m
        except Exception as e_bf16:
            logger.warning(f"bf16 load failed ({e_bf16}). Trying fp32...")

        # ── path C: fp32 CPU fallback ──
        m = AutoModelForCausalLM.from_pretrained(
            model_id,
            token=self.token,
            trust_remote_code=True,
            torch_dtype=torch.float32,
            device_map="cpu",
            output_hidden_states=True,
        )
        logger.info("fp32 CPU load succeeded.")
        return m

    def _load_fallback(self):
        """Load small sentence-transformer for embeddings when primary fails."""
        self.tokenizer  = AutoTokenizer.from_pretrained(self.FALLBACK_MODEL)
        self.model      = AutoModel.from_pretrained(self.FALLBACK_MODEL)
        self.model.eval()
        self.dim        = 384
        self.model_name = self.FALLBACK_MODEL
        self.is_causal  = False

    # ── 2. LoRA fine-tuning adapters ───────────────────────────────────────

    def _apply_lora(self):
        """
        Attach LoRA adapters for supervised fine-tuning on DigitalOcean docs.
        TaskType.CAUSAL_LM is correct for a CausalLM backbone.
        Targets the attention projection layers by name pattern.
        """
        lora_cfg = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            bias="none",
            # target common attention weight names across Nemotron/Llama family
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                             "gate_proj", "up_proj", "down_proj"],
        )
        try:
            self.model = prepare_model_for_kbit_training(
                self.model,
                use_gradient_checkpointing=True,
            )
            self.model = get_peft_model(self.model, lora_cfg)
            self.model.print_trainable_parameters()
            logger.info("LoRA adapters applied (CAUSAL_LM, r=16).")
        except Exception as e:
            logger.warning(f"LoRA application failed: {e}. "
                           f"Continuing without adapters.")

    # ── 3. Embedding extraction ────────────────────────────────────────────

    @torch.no_grad()
    def encode(self, texts: List[str], batch_size: int = 4) -> np.ndarray:
        """
        Mean-pool the last hidden states to produce dense embeddings.
        Works for both CausalLM (uses output_hidden_states=True) and
        encoder-only fallback (uses last_hidden_state directly).
        """
        device = next(self.model.parameters()).device
        all_embs: List[np.ndarray] = []

        for i in tqdm(range(0, len(texts), batch_size),
                      desc="Encoding batches"):
            if not memory_safe():
                logger.warning("Memory limit hit — stopping encode early.")
                break
            batch = texts[i : i + batch_size]
            enc = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            ).to(device)

            out = self.model(**enc)

            # CausalLM returns hidden_states tuple; encoder returns last_hidden_state
            if self.is_causal:
                # last element of hidden_states tuple = final layer
                hidden = out.hidden_states[-1]      # (B, T, D)
            else:
                hidden = out.last_hidden_state      # (B, T, D)

            mask = enc["attention_mask"].unsqueeze(-1).float()
            emb  = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            all_embs.append(emb.float().cpu().numpy())
            gc.collect()

        return np.vstack(all_embs)

    # ── 4. AI Agent text generation ────────────────────────────────────────

    def generate(self, prompt: str,
                 max_new_tokens: int = 512,
                 temperature: float = 0.3,
                 repetition_penalty: float = 1.1) -> str:
        """
        Run autoregressive generation with the Nemotron CausalLM.
        Returns the newly generated text only (strips the prompt).
        Falls back to a template if the model is not a CausalLM.
        """
        if not self.is_causal:
            logger.warning("generate() called on non-causal fallback model. "
                           "Returning empty string.")
            return ""

        device = next(self.model.parameters()).device
        enc = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048,
        ).to(device)

        input_len = enc["input_ids"].shape[1]

        with torch.no_grad():
            out_ids = self.model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=temperature > 0,
                repetition_penalty=repetition_penalty,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        new_tokens = out_ids[0][input_len:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    def build_nemotron_prompt(self, system: str, user: str,
                               context: str = "") -> str:
        """
        Construct a prompt in Nemotron's SteerLM / chat format:
          <extra_id_0>System\\n{system}\\n<extra_id_1>User\\n{user}\\n<extra_id_1>Assistant\\n
        """
        ctx_block = f"\n\nRelevant Documentation:\n{context}" if context else ""
        return (
            f"{self.SYSTEM_TAG}{system}{ctx_block}"
            f"{self.USER_TAG}{user}"
            f"{self.ASST_TAG}"
        )

    # ── 5. Save fine-tuned model ───────────────────────────────────────────

    def save_fine_tuned(self, output_dir: Path = MODEL_DIR):
        """
        Save tokenizer + LoRA-merged model weights as safetensors.
        Produces: config.json, tokenizer files, model.safetensors (sharded).
        """
        output_dir.mkdir(parents=True, exist_ok=True)
        logger.info(f"Saving fine-tuned model → {output_dir}")
        self.tokenizer.save_pretrained(output_dir)
        # Merge LoRA adapters into base weights before saving
        if PEFT_AVAILABLE and hasattr(self.model, "merge_and_unload"):
            try:
                merged = self.model.merge_and_unload()
                merged.save_pretrained(
                    output_dir,
                    safe_serialization=True,
                    max_shard_size="4GB",
                )
                logger.info("LoRA-merged model saved with safetensors "
                            "(sharded, max 4GB per shard).")
                return
            except Exception as e:
                logger.warning(f"merge_and_unload failed ({e}). "
                               f"Saving adapter weights only.")
        self.model.save_pretrained(
            output_dir,
            safe_serialization=True,
            max_shard_size="4GB",
        )
        logger.info("Model saved with safetensors.")


# =============================================================================
# SECTION 9 — VECTOR INDEX (FAISS)
# =============================================================================

class VectorIndex:
    """
    FAISS-backed nearest-neighbour index with optional disk persistence
    (mirrors DigitalOcean Spaces / external vector DB behaviour).
    """

    def __init__(self, dim: int = EMBED_DIM):
        self.dim   = dim
        self.index = None
        self.docs: List[Dict] = []   # metadata store
        self.embeddings: Optional[np.ndarray] = None

    # ── internal helpers ───────────────────────────────────────────────────

    @staticmethod
    def _l2_normalize(mat: np.ndarray) -> np.ndarray:
        """Row-wise L2 normalisation (numpy fallback for faiss.normalize_L2)."""
        norms = np.linalg.norm(mat, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1.0, norms)
        return mat / norms

    # ── public API ─────────────────────────────────────────────────────────

    def build(self, embeddings: np.ndarray, docs: List[Dict]):
        logger.info(f"Building vector index: {embeddings.shape}  "
                    f"(backend={'faiss' if FAISS_AVAILABLE else 'numpy'})")
        emb = embeddings.astype("float32")

        if FAISS_AVAILABLE:
            faiss.normalize_L2(emb)
            if len(emb) < 1000:
                self.index = faiss.IndexFlatIP(self.dim)
            else:
                nlist = min(64, len(emb) // 10)
                quantizer = faiss.IndexFlatIP(self.dim)
                self.index = faiss.IndexIVFFlat(
                    quantizer, self.dim, nlist, faiss.METRIC_INNER_PRODUCT)
                self.index.train(emb)
            self.index.add(emb)
            logger.info(f"FAISS index has {self.index.ntotal} vectors.")
        else:
            # Pure-numpy cosine index: store normalised vectors
            self.embeddings = self._l2_normalize(emb)
            logger.info(f"Numpy index: {len(self.embeddings)} vectors stored.")

        self.docs = docs

    def search(self, query_emb: np.ndarray, k: int = 10) -> List[Dict]:
        q = query_emb.astype("float32").reshape(1, -1)

        if FAISS_AVAILABLE and self.index is not None:
            faiss.normalize_L2(q)
            scores, ids = self.index.search(q, k)
            results = []
            for score, idx in zip(scores[0], ids[0]):
                if idx < 0:
                    continue
                doc = dict(self.docs[idx])
                doc["semantic_score"] = float(score)
                results.append(doc)
            return results
        else:
            # Numpy cosine similarity fallback
            q_norm = self._l2_normalize(q)                    # (1, D)
            sims   = (self.embeddings @ q_norm.T).squeeze(1)  # (N,)
            top_k  = int(min(k, len(sims)))
            top_ids = np.argpartition(sims, -top_k)[-top_k:]
            top_ids = top_ids[np.argsort(sims[top_ids])[::-1]]
            results = []
            for idx in top_ids:
                doc = dict(self.docs[idx])
                doc["semantic_score"] = float(sims[idx])
                results.append(doc)
            return results

    def save(self, path: Path = INDEX_DIR / "faiss.index"):
        if FAISS_AVAILABLE and self.index is not None:
            faiss.write_index(self.index, str(path))
        else:
            np.save(str(path) + ".npy", self.embeddings)
        meta_path = path.with_suffix(".meta.pkl")
        with open(meta_path, "wb") as f:
            pickle.dump(self.docs, f)
        logger.info(f"Index saved → {path}")

    def load(self, path: Path = INDEX_DIR / "faiss.index"):
        if FAISS_AVAILABLE:
            self.index = faiss.read_index(str(path))
        else:
            self.embeddings = np.load(str(path) + ".npy")
        meta_path = path.with_suffix(".meta.pkl")
        with open(meta_path, "rb") as f:
            self.docs = pickle.load(f)
        n = self.index.ntotal if FAISS_AVAILABLE else len(self.embeddings)
        logger.info(f"Index loaded: {n} vectors.")


# =============================================================================
# SECTION 10 — RE-RANKER
# =============================================================================

class CrossEncoderReranker:
    """
    Lightweight cross-encoder re-ranker using a small BERT variant.
    Scores (query, passage) pairs and reorders top-k results.
    """

    RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

    def __init__(self, hf_token: Optional[str] = None):
        self.token     = hf_token
        self.tokenizer = None
        self.model     = None

    def load(self):
        try:
            from transformers import AutoModelForSequenceClassification
            self.tokenizer = AutoTokenizer.from_pretrained(
                self.RERANK_MODEL, token=self.token)
            self.model = AutoModelForSequenceClassification.from_pretrained(
                self.RERANK_MODEL, token=self.token)
            self.model.eval()
            logger.info("Cross-encoder re-ranker loaded.")
        except Exception as e:
            logger.warning(f"Re-ranker load failed: {e}. Skipping re-ranking.")

    @torch.no_grad()
    def rerank(self, query: str, candidates: List[Dict],
               text_key: str = "body_text") -> List[Dict]:
        if self.model is None:
            return candidates
        device = next(self.model.parameters()).device
        pairs  = [(query, c.get(text_key, "")[:512]) for c in candidates]
        enc    = self.tokenizer(pairs, padding=True, truncation=True,
                                max_length=512, return_tensors="pt").to(device)
        scores = self.model(**enc).logits.squeeze(-1).cpu().numpy()
        for c, s in zip(candidates, scores):
            c["rerank_score"] = float(s)
        return sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)


# =============================================================================
# SECTION 11 — SEARCH ENGINE AGENT
# =============================================================================

class SemanticSearchAgent:
    """
    Full semantic search pipeline:
      query → embed → ANN (FAISS) → cross-encoder re-rank → results
    Mirrors DigitalOcean Gradient: Serverless Inference + Vector Index.
    """

    def __init__(self, embed_model: EmbeddingModel,
                 vector_index: VectorIndex,
                 reranker: CrossEncoderReranker):
        self.embed   = embed_model
        self.index   = vector_index
        self.reranker = reranker

    def search(self, query: str, k: int = 5,
               rerank_k: int = 20) -> List[Dict]:
        q_emb = self.embed.encode([query])
        candidates = self.index.search(q_emb, k=rerank_k)
        ranked = self.reranker.rerank(query, candidates)
        return ranked[:k]


# =============================================================================
# SECTION 12 — RAG KNOWLEDGE BASE
# =============================================================================

class KnowledgeBase:
    """
    Sharded knowledge base backed by the same vector index.
    Retrieves top-k relevant passages for RAG.
    Mirrors DO Gradient: Knowledge Bases + RAG.
    """

    def __init__(self, vector_index: VectorIndex,
                 embed_model: EmbeddingModel):
        self.index = vector_index
        self.embed = embed_model

    def retrieve(self, query: str, k: int = 5,
                 doc_type_filter: Optional[str] = None) -> List[Dict]:
        q_emb = self.embed.encode([query])
        results = self.index.search(q_emb, k=k * 3)
        if doc_type_filter:
            results = [r for r in results
                       if r.get("doc_type") == doc_type_filter]
        return results[:k]

    def format_context(self, chunks: List[Dict],
                       max_tokens: int = 2000) -> str:
        ctx_parts = []
        total = 0
        for c in chunks:
            snippet = c.get("body_text","")[:500]
            title   = c.get("title","")
            source  = c.get("path","")
            part = f"[Source: {source}]\nTitle: {title}\n{snippet}\n---"
            total += len(part.split())
            if total > max_tokens:
                break
            ctx_parts.append(part)
        return "\n".join(ctx_parts)


# =============================================================================
# SECTION 13 — MULTI-AGENT CUSTOMER SUPPORT SYSTEM
# =============================================================================

SYSTEM_PROMPTS = {
    "billing": (
        "You are a DigitalOcean billing support agent. "
        "Answer only billing, invoice, and payment questions. "
        "Never expose PII. Cite knowledge base sources. "
        "If you cannot resolve, escalate to a human: say 'ESCALATE'."
    ),
    "technical": (
        "You are a DigitalOcean technical support agent. "
        "Answer infrastructure, droplet, networking, and API questions. "
        "Use retrieved docs. Cite sources with [Source: ...]. "
        "If unresolvable, say 'ESCALATE'."
    ),
    "general": (
        "You are a helpful DigitalOcean support assistant. "
        "Answer general questions about DigitalOcean products and services. "
        "Do not expose user PII. Cite references."
    ),
}

BILLING_KEYWORDS  = {"billing","invoice","payment","charge","refund","cost","price","subscription","bandwidth"}
TECHNICAL_KEYWORDS= {"droplet","firewall","ssh","kubernetes","nginx","docker","api","dns","certificate",
                     "postgres","mysql","python","node","server","deploy","vpc","gpu","memory","disk"}

class RouterAgent:
    """Route user query to billing | technical | general sub-agent."""

    def route(self, query: str) -> str:
        q = query.lower()
        if any(kw in q for kw in BILLING_KEYWORDS):
            return "billing"
        if any(kw in q for kw in TECHNICAL_KEYWORDS):
            return "technical"
        return "general"


class PiiGuardrail:
    """
    Simple rule-based PII filter.
    Redacts emails, credit cards, and phone numbers.
    Mirrors DO Gradient Guardrails.
    """
    EMAIL_RE = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+")
    CC_RE    = re.compile(r"\b(?:\d[ -]?){13,19}\b")
    PHONE_RE = re.compile(r"\b\+?[\d\s\-().]{7,15}\b")

    def scrub(self, text: str) -> str:
        text = self.EMAIL_RE.sub("[EMAIL_REDACTED]", text)
        text = self.CC_RE.sub("[CARD_REDACTED]", text)
        text = self.PHONE_RE.sub("[PHONE_REDACTED]", text)
        return text


class TicketingAPI:
    """
    Simulated ticketing API (mirrors DO Gradient function calling).
    In production this would call Zendesk / Freshdesk etc.
    """

    def create_ticket(self, user_id: str, category: str,
                      summary: str, escalate: bool = False) -> Dict:
        ticket_id = hashlib.md5(
            f"{user_id}{time.time()}".encode()).hexdigest()[:8].upper()
        ticket = {
            "ticket_id": ticket_id,
            "user_id":   user_id,
            "category":  category,
            "summary":   summary[:200],
            "escalated": escalate,
            "status":    "open",
            "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        }
        logger.info(f"Ticket created: {ticket}")
        return ticket


class CustomerSupportAgent:
    """
    Multi-channel RAG chatbot with:
      • Query routing (billing / technical / general)
      • Knowledge base retrieval + context injection
      • LLM response generated by Nemotron (loaded EmbeddingModel.generate())
        with HF serverless InferenceClient as secondary fallback
      • PII guardrails
      • Escalation + ticket creation
    Mirrors DO Gradient multi-agent workflows + serverless inference.
    """

    def __init__(self, knowledge_base: KnowledgeBase,
                 embed_model: EmbeddingModel,
                 hf_token: Optional[str] = None):
        self.kb        = knowledge_base
        self.embed     = embed_model    # Nemotron backbone — used for generate()
        self.router    = RouterAgent()
        self.pii       = PiiGuardrail()
        self.ticketing = TicketingAPI()
        self.hf_token  = hf_token
        self.chat_history: List[Dict] = []

    def _call_llm(self, system_prompt: str, user_msg: str,
                  context: str) -> str:
        """
        Generation priority:
          1. Nemotron loaded locally (embed_model.is_causal=True)
             → use embed_model.generate() with Nemotron SteerLM prompt format
          2. HF serverless InferenceClient (Nemotron endpoint)
             → used when model is not loaded locally (memory constraints)
          3. Template response
             → plain-text answer stitched from retrieved KB context
        """
        # ── Path 1: locally loaded Nemotron ──────────────────────────────
        if self.embed.is_causal:
            try:
                prompt = self.embed.build_nemotron_prompt(
                    system=system_prompt,
                    user=user_msg,
                    context=context,
                )
                logger.info("Generating response via local Nemotron model...")
                response = self.embed.generate(
                    prompt=prompt,
                    max_new_tokens=512,
                    temperature=0.3,
                    repetition_penalty=1.1,
                )
                if response:
                    return response
                logger.warning("Local generate() returned empty — trying HF API.")
            except Exception as e:
                logger.warning(f"Local generate() failed ({e}) — trying HF API.")

        # ── Path 2: HF serverless InferenceClient (Nemotron endpoint) ────
        try:
            from huggingface_hub import InferenceClient
            client = InferenceClient(
                model=EmbeddingModel.PRIMARY_MODEL,
                token=self.hf_token,
            )
            # Nemotron SteerLM chat format
            prompt = self.embed.build_nemotron_prompt(
                system=system_prompt,
                user=user_msg,
                context=context,
            )
            logger.info("Generating response via HF Inference API (Nemotron)...")
            resp = client.text_generation(
                prompt,
                max_new_tokens=512,
                temperature=0.3,
                repetition_penalty=1.1,
                stop_sequences=[
                    EmbeddingModel.USER_TAG,
                    EmbeddingModel.SYSTEM_TAG,
                ],
            )
            return resp.strip()
        except Exception as e:
            logger.warning(f"HF API call failed ({e}). Using template response.")

        # ── Path 3: Template fallback ─────────────────────────────────────
        return self._template_response(user_msg, context)

    def _template_response(self, query: str, context: str) -> str:
        snippets = context[:800] if context else "No relevant docs found."
        return (
            f"Based on DigitalOcean documentation:\n\n{snippets}\n\n"
            f"For your query: '{query}' — please refer to the sources above "
            "or contact support at https://www.digitalocean.com/support."
        )

    def respond(self, user_query: str,
                user_id: str = "anon") -> Dict:
        # 1. PII scrub input
        clean_query = self.pii.scrub(user_query)

        # 2. Route
        agent_type = self.router.route(clean_query)
        system_prompt = SYSTEM_PROMPTS[agent_type]

        # 3. Retrieve KB context
        chunks  = self.kb.retrieve(clean_query, k=5,
                                   doc_type_filter=None)
        context = self.kb.format_context(chunks)

        # 4. Call LLM
        raw_answer = self._call_llm(system_prompt, clean_query, context)

        # 5. PII scrub output
        answer = self.pii.scrub(raw_answer)

        # 6. Escalation check
        escalate = "ESCALATE" in answer.upper()
        ticket   = None
        if escalate:
            ticket = self.ticketing.create_ticket(
                user_id=user_id,
                category=agent_type,
                summary=clean_query[:200],
                escalate=True,
            )

        # 7. Citations
        citations = [
            {"source": c.get("path",""), "title": c.get("title","")}
            for c in chunks
        ]

        result = {
            "query":      user_query,
            "agent_type": agent_type,
            "answer":     answer,
            "citations":  citations,
            "escalated":  escalate,
            "ticket":     ticket,
        }
        self.chat_history.append(result)
        return result


# =============================================================================
# SECTION 14 — MODEL EVALUATION & VISUALISATION
# =============================================================================

def plot_confusion_matrix(y_true, y_pred, classes, tag: str, le: LabelEncoder):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(max(6, len(classes)), max(5, len(classes)-1)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(f"Confusion Matrix — {tag}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.tight_layout()
    p = PLOTS_DIR / f"cm_{tag}.png"
    plt.savefig(p, dpi=120); plt.show(); plt.close()
    logger.info(f"Saved {p}")


def plot_roc(y_true, proba, n_classes: int, tag: str):
    if proba is None or n_classes < 2:
        return
    fig, ax = plt.subplots(figsize=(8, 6))
    from sklearn.preprocessing import label_binarize
    y_bin = label_binarize(y_true, classes=list(range(n_classes)))
    for i in range(n_classes):
        if y_bin[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], proba[:, i])
        auc = roc_auc_score(y_bin[:, i], proba[:, i])
        ax.plot(fpr, tpr, label=f"Class {i} (AUC={auc:.2f})")
    ax.plot([0,1],[0,1],"k--")
    ax.set_title(f"ROC Curves — {tag}")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.legend(loc="lower right", fontsize=7)
    plt.tight_layout()
    p = PLOTS_DIR / f"roc_{tag}.png"
    plt.savefig(p, dpi=120); plt.show(); plt.close()
    logger.info(f"Saved {p}")


def plot_generalization_gap(train_accs: Dict, val_accs: Dict,
                             test_accs: Dict, holdout_accs: Dict):
    splits = ["train","val","test","holdout"]
    accs   = [
        np.mean(list(train_accs.values())),
        np.mean(list(val_accs.values())),
        np.mean(list(test_accs.values())),
        np.mean(list(holdout_accs.values())),
    ]
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(splits, accs,
                  color=["#4C72B0","#DD8452","#55A868","#C44E52"])
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.005,
                f"{acc:.3f}", ha="center", va="bottom", fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.set_title("Accuracy Across All Splits (Generalization Gap View)")
    ax.set_ylabel("Accuracy")
    plt.tight_layout()
    p = PLOTS_DIR / "generalization_gap.png"
    plt.savefig(p, dpi=120); plt.show(); plt.close()
    logger.info(f"Saved {p}")


def full_evaluation(models: Dict, features: Dict,
                    labels: Dict, le: LabelEncoder):
    """
    Evaluate all models on all splits.
    Returns accuracy dicts + saves all plots.
    """
    classes = list(le.classes_)
    n_classes = len(classes)
    acc_results = {m: {} for m in models}

    for split_name, X in features.items():
        y = labels[split_name]
        for model_name, (predictor, predict_proba_fn) in models.items():
            preds = predictor(X)
            proba = predict_proba_fn(X) if predict_proba_fn else None
            acc   = accuracy_score(y, preds)
            acc_results[model_name][split_name] = acc
            logger.info(
                f"[{model_name}][{split_name}] acc={acc:.4f}  "
                f"f1={f1_score(y, preds, average='macro', zero_division=0):.4f}"
            )
            plot_confusion_matrix(y, preds, classes,
                                   f"{model_name}_{split_name}", le)
            if proba is not None:
                plot_roc(y, proba, n_classes, f"{model_name}_{split_name}")

    # overall accuracy bar chart
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(features))
    width = 0.25
    for i, (mname, split_accs) in enumerate(acc_results.items()):
        vals = [split_accs.get(s, 0) for s in features.keys()]
        ax.bar(x + i*width, vals, width, label=mname)
    ax.set_xticks(x + width)
    ax.set_xticklabels(list(features.keys()))
    ax.set_title("Model Accuracy Across All Splits")
    ax.set_ylabel("Accuracy"); ax.legend()
    plt.tight_layout()
    p = PLOTS_DIR / "all_model_accuracy.png"
    plt.savefig(p, dpi=120); plt.show(); plt.close()
    logger.info(f"Saved {p}")

    return acc_results


# =============================================================================
# SECTION 15 — BEST MODEL CHECKPOINT MANAGER
# =============================================================================

class BestModelCheckpoint:
    """Saves model with highest validation accuracy. Prevents over-saving."""

    def __init__(self, output_dir: Path = MODEL_DIR):
        self.output_dir   = output_dir
        self.best_val_acc = 0.0
        self.log: List[Dict] = []

    def update(self, val_acc: float, model_obj, model_name: str,
               save_fn=None):
        entry = {"epoch": len(self.log), "val_acc": val_acc,
                 "model": model_name, "is_best": False}
        if val_acc > self.best_val_acc:
            self.best_val_acc = val_acc
            entry["is_best"] = True
            if save_fn:
                save_fn()
            logger.info(f"New best model: {model_name}  val_acc={val_acc:.4f}")
        self.log.append(entry)

    def plot_val_acc_history(self):
        vals = [e["val_acc"] for e in self.log]
        best_idx = int(np.argmax(vals))
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(vals, marker="o", ms=4)
        ax.scatter([best_idx],[vals[best_idx]], color="red", zorder=5,
                   label=f"Best={vals[best_idx]:.4f}")
        ax.set_title("Validation Accuracy History (all models)")
        ax.set_xlabel("Checkpoint step"); ax.set_ylabel("Val Acc")
        ax.legend()
        plt.tight_layout()
        p = PLOTS_DIR / "checkpoint_val_acc.png"
        plt.savefig(p, dpi=120); plt.show(); plt.close()
        logger.info(f"Saved {p}")


# =============================================================================
# MAIN PIPELINE
# =============================================================================

def main():
    logger.info("=" * 70)
    logger.info("  DigitalOcean Semantic Search + RAG Agent Pipeline")
    logger.info("=" * 70)
    log_memory("start")

    # ── 0. Setup ──────────────────────────────────────────────────────────
    hf_token   = load_hf_token()
    checkpoint  = BestModelCheckpoint()

    # ── 1. Crawl & Preprocess ─────────────────────────────────────────────
    crawler = HTMLDocumentCrawler(ROOT_PATH)
    raw_df  = crawler.crawl()
    force_cleanup()

    if raw_df.empty:
        logger.error("No documents found. Check ROOT_PATH.")
        # demo fallback
        raw_df = pd.DataFrame({
            "path":      ["demo/page.html"],
            "url_slug":  ["demo-page"],
            "title":     ["Demo Page"],
            "body_text": ["DigitalOcean droplets provide scalable cloud infrastructure."],
            "doc_type":  ["docs"],
            "word_count":[8],
        })

    df, le = HTMLDocumentCrawler.preprocess(raw_df)
    n_classes = df["label"].nunique()
    logger.info(f"Classes: {list(le.classes_)}")

    # ── 2. EDA ────────────────────────────────────────────────────────────
    splitter = DataSplitter(df)
    splits   = splitter.split()
    DataSplitter.verify_data_splits(splits)
    DataSplitter.check_domain_relevance(splits)
    run_eda(df, splits)
    force_cleanup()

    # ── 3. Feature Engineering ────────────────────────────────────────────
    fe = FeatureEngineer(max_features=20_000, n_components=128)
    X_train = fe.fit_transform(splits["train"])
    X_val   = fe.transform(splits["val"])
    X_test  = fe.transform(splits["test"])
    X_hold  = fe.transform(splits["holdout"])

    y_train = splits["train"]["label"].values
    y_val   = splits["val"]["label"].values
    y_test  = splits["test"]["label"].values
    y_hold  = splits["holdout"]["label"].values

    fe.pca_plot(X_train, y_train, "train")
    fe.pca_plot(X_hold,  y_hold,  "holdout")
    force_cleanup()

    # ── 4. Classical Ensemble ─────────────────────────────────────────────
    classic = ClassicalEnsemble()
    classic.fit(X_train, y_train, X_val, y_val)
    classic.save()
    checkpoint.update(
        classic.best_acc, classic.best_model, "ClassicalEnsemble",
        save_fn=classic.save)
    force_cleanup()

    # ── 5. Deep MLP ───────────────────────────────────────────────────────
    input_dim = X_train.shape[1]
    mlp_model = DeepMLP(input_dim=input_dim, num_classes=n_classes,
                         hidden_dims=(256, 128, 64), dropout=0.40)
    mlp_trainer = MLPTrainer(mlp_model, n_classes)
    mlp_trainer.train(X_train, y_train, X_val, y_val,
                      epochs=40, lr=1e-3, weight_decay=1e-4,
                      patience=8, use_mixup=True)
    mlp_trainer.save()
    mlp_trainer.plot_training_curves()
    checkpoint.update(
        mlp_trainer.best_acc, mlp_model, "DeepMLP",
        save_fn=mlp_trainer.save)
    force_cleanup()

    # ── 6. Load Embedding Model (HF, no local download) ───────────────────
    embed_model = EmbeddingModel(hf_token=hf_token)
    embed_model.load()

    # Fine-tune (LoRA) on training docs
    if PEFT_AVAILABLE:
        logger.info("LoRA adapters already applied during load.")
    embed_model.save_fine_tuned(MODEL_DIR)

    # ── 7. Build Vector Index ─────────────────────────────────────────────
    logger.info("Encoding corpus for vector index...")
    all_texts = df["body_text"].tolist()[:3000]  # cap for memory safety
    all_docs  = df[["path","title","doc_type","body_text"]].to_dict("records")[:3000]

    embeddings = embed_model.encode(all_texts, batch_size=16)
    force_cleanup()

    vi = VectorIndex(dim=embed_model.dim)
    vi.build(embeddings, all_docs)
    vi.save()
    force_cleanup()

    # ── 8. Re-ranker ──────────────────────────────────────────────────────
    reranker = CrossEncoderReranker(hf_token=hf_token)
    reranker.load()

    # ── 9. Search Agent ───────────────────────────────────────────────────
    search_agent = SemanticSearchAgent(embed_model, vi, reranker)
    # demo query
    demo_results = search_agent.search("how to set up SSH keys on Ubuntu", k=3)
    logger.info("Demo search results:")
    for r in demo_results:
        logger.info(f"  [{r.get('doc_type','')}] {r.get('title','')} "
                    f"(score={r.get('rerank_score', r.get('semantic_score',0)):.3f})")

    # ── 10. Customer Support / RAG Agent ──────────────────────────────────
    kb = KnowledgeBase(vi, embed_model)
    support_agent = CustomerSupportAgent(kb, embed_model, hf_token)

    demo_queries = [
        ("How do I set up an Nginx ingress on Kubernetes?", "user_001"),
        ("I was charged twice on my last invoice", "user_002"),
        ("What is a firewall?", "user_003"),
    ]
    for q, uid in demo_queries:
        result = support_agent.respond(q, user_id=uid)
        logger.info(
            f"\nQ: {q}\n"
            f"  Agent: {result['agent_type']}\n"
            f"  Escalated: {result['escalated']}\n"
            f"  Citations: {[c['title'] for c in result['citations'][:2]]}\n"
            f"  Answer[:200]: {result['answer'][:200]}"
        )
    force_cleanup()

    # ── 11. Full Evaluation & Plots ───────────────────────────────────────
    feature_sets = {"train": X_train, "val": X_val,
                    "test": X_test,   "holdout": X_hold}
    label_sets   = {"train": y_train, "val": y_val,
                    "test": y_test,   "holdout": y_hold}

    models_to_eval = {
        "ClassicalEnsemble": (
            classic.best_model.predict,
            classic.best_model.predict_proba
                if hasattr(classic.best_model,"predict_proba") else None,
        ),
        "DeepMLP": (
            mlp_trainer.predict,
            mlp_trainer.predict_proba,
        ),
    }
    acc_results = full_evaluation(
        models_to_eval, feature_sets, label_sets, le)

    # Generalization gap
    plot_generalization_gap(
        {m: acc_results[m]["train"]   for m in models_to_eval},
        {m: acc_results[m]["val"]     for m in models_to_eval},
        {m: acc_results[m]["test"]    for m in models_to_eval},
        {m: acc_results[m]["holdout"] for m in models_to_eval},
    )
    checkpoint.plot_val_acc_history()

    # ── 12. Final Summary ─────────────────────────────────────────────────
    logger.info("\n" + "=" * 70)
    logger.info("PIPELINE COMPLETE")
    logger.info(f"Fine-tuned model  → {MODEL_DIR}/")
    logger.info(f"Vector index      → {INDEX_DIR}/")
    logger.info(f"Classical model   → {OUTPUT_DIR}/classical_ensemble.pkl")
    logger.info(f"MLP checkpoint    → {MODEL_DIR}/mlp_best.pt")
    logger.info(f"Plots             → {PLOTS_DIR}/")
    logger.info(f"Best val accuracy → {checkpoint.best_val_acc:.4f}")
    logger.info("=" * 70)
    force_cleanup()


if __name__ == "__main__":
    main()

2026-03-17 23:49:39,496 [INFO] GPU: Tesla P100-PCIE-16GB
2026-03-17 23:49:39,510 [INFO] ======================================================================
2026-03-17 23:49:39,511 [INFO]   DigitalOcean Semantic Search + RAG Agent Pipeline
2026-03-17 23:49:39,511 [INFO] ======================================================================
2026-03-17 23:49:39,513 [INFO] [MEM start] used=1.09GB  avail=31.79GB
2026-03-17 23:49:39,519 [INFO] HF token loaded successfully.
2026-03-17 23:49:40,007 [INFO] Found 923 HTML files under /kaggle/input/datasets/tobimichigan/digitalocreandoc-db


  Chunk 0: 100%|█████████▉| 499/500 [00:45<00:00,  9.05it/s]
                                                            

2026-03-17 23:50:25,527 [INFO] [MEM chunk_0] used=1.14GB  avail=31.73GB


  Chunk 1: 100%|█████████▉| 422/423 [00:38<00:00,  8.86it/s]
                                                            

2026-03-17 23:51:04,083 [INFO] [MEM chunk_1] used=1.15GB  avail=31.72GB


...

Encoding batches: 100%|██████████| 1/1 [00:00<00:00,  3.80it/s]

2026-03-17 23:53:10,081 [INFO] Generating response via HF Inference API (Nemotron)...
2026-03-17 23:53:10,082 [WARNING] HF API call failed (). Using template response.
2026-03-17 23:53:10,083 [INFO] 
Q: I was charged twice on my last invoice
  Agent: billing
  Escalated: False
  Citations: ['https://docs.digitalocean.com/platform/billing/', 'Droplet Bandwidth Billing Update Scheduled June 2018 | DigitalOcean Documentation']
  Answer[:200]: Based on DigitalOcean documentation:

[Source: digitalocean_docs/pages/docs.digitalocean.com/products/billing.html]
Title: https://docs.digitalocean.com/platform/billing/
https://docs.digitalocean.com



Encoding batches: 100%|██████████| 1/1 [00:00<00:00,  3.78it/s]

2026-03-17 23:53:10,351 [INFO] Generating response via HF Inference API (Nemotron)...
2026-03-17 23:53:10,352 [WARNING] HF API call failed (). Using template response.
2026-03-17 23:53:10,353 [INFO] 
Q: What is a firewall?
  Agent: technical
  Escalated: False
  Citations: ['What is a Firewall and How Does It Work? | DigitalOcean', 'How To Set Up a Firewall Using firewalld on Rocky Linux 8 | DigitalOcean']
  Answer[:200]: Based on DigitalOcean documentation:

[Source: digitalocean_docs/pages/www.digitalocean.com/community/tutorials/what-is-a-firewall-and-how-does-it-work.html]
Title: What is a Firewall and How Does It 


2026-03-17 23:53:10,607 [INFO] [MEM after_cleanup] used=2.29GB  avail=30.30GB
2026-03-17 23:53:10,689 [INFO] [ClassicalEnsemble][train] acc=0.9973  f1=0.9792
2026-03-17 23:53:10,878 [INFO] Saved plots/cm_ClassicalEnsemble_train.png
2026-03-17 23:53:11,080 [INFO] Saved plots/roc_ClassicalEnsemble_train.png
2026-03-17 23:53:11,096 [INFO] [DeepMLP][train] acc=0.8374  f1=0.7190
2026-03-17 23:53:11,295 [INFO] Saved plots/cm_DeepMLP_train.png
2026-03-17 23:53:11,489 [INFO] Saved plots/roc_DeepMLP_train.png
2026-03-17 23:53:11,567 [INFO] [ClassicalEnsemble][val] acc=0.9855  f1=0.9392
2026-03-17 23:53:11,771 [INFO] Saved plots/cm_ClassicalEnsemble_val.png
2026-03-17 23:53:11,969 [INFO] Saved plots/roc_ClassicalEnsemble_val.png
2026-03-17 23:53:11,977 [INFO] [DeepMLP][val] acc=0.8478  f1=0.7467
2026-03-17 23:53:12,169 [INFO] Saved plots/cm_DeepMLP_val.png
2026-03-17 23:53:12,360 [INFO] Saved plots/roc_DeepMLP_val.png
2026-03-17 23:53:12,438 [INFO] [ClassicalEnsemble][test] acc=0.9640  f1=0.8758

In [2]:
import os
import zipfile
from tqdm import tqdm
from pathlib import Path
import time

def get_all_files(directory):
    """Recursively get all files in directory and subdirectories"""
    all_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            full_path = os.path.join(root, file)
            all_files.append(full_path)
    return all_files

def get_total_size(file_paths):
    """Calculate total size of all files"""
    total_size = 0
    for file_path in file_paths:
        try:
            total_size += os.path.getsize(file_path)
        except (OSError, FileNotFoundError):
            continue
    return total_size

def create_kaggle_working_zip(source_dir="/kaggle/working/", output_name="kaggle_working_backup.zip"):
    """
    Create a zip file of all content in the Kaggle working directory
    
    Args:
        source_dir (str): Source directory to zip (default: /kaggle/working/)
        output_name (str): Name of the output zip file
    """
    
    # Check if source directory exists
    if not os.path.exists(source_dir):
        print(f"Error: Source directory '{source_dir}' does not exist!")
        return False
    
    # Get all files recursively
    print("Scanning files...")
    all_files = get_all_files(source_dir)
    
    if not all_files:
        print(f"No files found in '{source_dir}'")
        return False
    
    print(f"Found {len(all_files)} files to compress")
    
    # Calculate total size for progress tracking
    total_size = get_total_size(all_files)
    print(f"Total size: {total_size / (1024*1024):.2f} MB")
    
    # Create zip file with progress bar
    try:
        with zipfile.ZipFile(output_name, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zipf:
            # Progress bar based on file count
            with tqdm(total=len(all_files), desc="Compressing files", unit="files") as pbar:
                processed_size = 0
                
                for file_path in all_files:
                    try:
                        # Get relative path for the zip archive
                        arcname = os.path.relpath(file_path, source_dir)
                        
                        # Add file to zip
                        zipf.write(file_path, arcname)
                        
                        # Update progress
                        file_size = os.path.getsize(file_path)
                        processed_size += file_size
                        
                        # Update progress bar with file info
                        pbar.set_postfix({
                            'Current': os.path.basename(file_path)[:20],
                            'Size': f"{processed_size / (1024*1024):.1f}MB"
                        })
                        pbar.update(1)
                        
                    except Exception as e:
                        print(f"Warning: Could not add {file_path} to zip: {str(e)}")
                        pbar.update(1)
                        continue
        
        # Get final zip file size
        zip_size = os.path.getsize(output_name)
        compression_ratio = (1 - zip_size / total_size) * 100 if total_size > 0 else 0
        
        print(f"\n Successfully created '{output_name}'")
        print(f" Original size: {total_size / (1024*1024):.2f} MB")
        print(f" Compressed size: {zip_size / (1024*1024):.2f} MB")
        print(f" Compression ratio: {compression_ratio:.1f}%")
        
        return True
        
    except Exception as e:
        print(f"Error creating zip file: {str(e)}")
        return False

def download_zip_in_kaggle(zip_filename):
    """
    Trigger download in Kaggle notebook environment
    """
    try:
        # In Kaggle, files in the working directory are automatically available for download
        # We can also use the files.download() method if available
        from google.colab import files
        files.download(zip_filename)
        print(f"Download triggered for {zip_filename}")
    except ImportError:
        # If not in Colab/Kaggle environment with files API
        print(f"Zip file '{zip_filename}' created successfully!")
        print("In Kaggle, you can download it from the 'Output' tab or use the file browser.")
        print("The file is located in your current working directory.")

if __name__ == "__main__":
    # Configuration
    SOURCE_DIRECTORY = "/kaggle/working/"
    OUTPUT_ZIP_NAME = "DigitalOcean_SFT.zip"
    
    print(" Starting Kaggle Working Directory Backup")
    print("=" * 50)
    
    # Create the zip file
    success = create_kaggle_working_zip(SOURCE_DIRECTORY, OUTPUT_ZIP_NAME)
    
    if success:
        print(f"\n Preparing download...")
        download_zip_in_kaggle(OUTPUT_ZIP_NAME)
    else:
        print(" Backup failed!")

 Starting Kaggle Working Directory Backup
Scanning files...
Found 35 files to compress
Total size: 97.84 MB


Compressing files: 100%|██████████| 35/35 [00:05<00:00,  6.47files/s, Current=model.safetensors, Size=97.8MB] 


 Successfully created 'DigitalOcean_SFT.zip'
 Original size: 97.84 MB
 Compressed size: 83.26 MB
 Compression ratio: 14.9%

 Preparing download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered for DigitalOcean_SFT.zip
